Upload `custom_dataset_images.zip` when prompted (only needed in Colab).

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()
zipfile.ZipFile(list(uploaded.keys())[0]).extractall('.')

In [ ]:
import os
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import datasets, transforms
from torchvision import models

import cv2
import matplotlib.pyplot as plt

In [ ]:
os.listdir('data/dataset')

In [ ]:
root = 'data/dataset'

In [ ]:
data_dict = {'X': [], 'y': []}

for cls in os.listdir(root):
    cls_path = os.path.join(root, cls)
    for img in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img)
        data_dict['X'].append(img_path)
        data_dict['y'].append(int(cls))

In [ ]:
pd.DataFrame(data_dict)

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, data_dict: dict, transforms=None):
        self.data_dict = data_dict
        self.transforms = transforms

    def __getitem__(self, idx):
        x = cv2.imread(self.data_dict['X'][idx])
        y = self.data_dict['y'][idx]
        if self.transforms:
            x = self.transforms(x)
            y = torch.tensor(y)
        return x, y

    def __len__(self):
        return len(self.data_dict['X'])

In [ ]:
trans = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize(size=(300, 300)),
    transforms.Normalize(mean=[0, 0, 0], std=[1, 1, 1])
])

cd = CustomDataset(data_dict, trans)

In [ ]:
for i in cd:
    print(i[0].shape)

In [ ]:
model = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

In [ ]:
x = cd[0][0].unsqueeze(dim=0)

In [ ]:
y_pred = model(x)

In [ ]:
torch.argmax(y_pred)

In [ ]:
classes = models.ResNet50_Weights.DEFAULT.meta

In [ ]:
classes.keys()

In [ ]:
classes['categories'][torch.argmax(y_pred)]

In [ ]:
plt.imshow(cd[0][0].permute(1, 2, 0))